In [1]:
#r "nuget: Microsoft.Z3, 5.1.0"
using System;
using System.Collections.Generic;
using System.Linq;
using Microsoft.Z3;

// Parametres
bool BATCH_MODE = true;
int SEED = 42;
int N_TRIES_PAR_TAILLE = 12;

Console.WriteLine($"Configuration : batch={BATCH_MODE}, seed={SEED}, essais_par_taille={N_TRIES_PAR_TAILLE}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 5.1.0

Configuration : batch=True, seed=42, essais_par_taille=12


# Frontieres-01 : Coloration d'arêtes et conjecture de Tutte (C#)

**Navigation** : [Version Python](Frontieres-01-EdgeColoring-Tutte-Python.ipynb) | [Version Rust](Frontieres-01-EdgeColoring-Tutte-Rust.ipynb) | [Série Search](../README.md)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Définir** la coloration d'arêtes d'un graphe, son indice chromatique et les classes de Vizing
2. **Formuler** la 3-coloration d'arêtes d'un graphe cubique comme un CSP résolu par le solveur Z3 (chaque couleur = un couplage parfait)
3. **Construire** des familles témoins : le graphe de Petersen (plus petit snark), un graphe cubique à pont, les échelles de Möbius (non planaires mais apex)
4. **Vérifier empiriquement** un théorème récent (arXiv 2608.22870, 2026) : tout graphe cubique **sans pont et apex** admet une 3-coloration d'arêtes

Ce notebook prolonge [App-2 : Coloration de Graphes](../Part3-Cas-pratiques/App-02-GraphColoring-Python.ipynb), qui traitait la coloration de **sommets** — ici, ce sont les **arêtes** que l'on colore.

C'est le portage C# (kernel .NET Interactive) de [Frontieres-01 en Python](Frontieres-01-EdgeColoring-Tutte-Python.ipynb) : CP-SAT y est remplacé par le solveur **Z3** (`Microsoft.Z3` via NuGet), et les outils networkx par des implémentations du carnet (ponts par parcours lowlink, planarité par le test gauche-droite de de Fraysseix–Rosenstiehl, échantillonnage par le modèle d'appariement). Les sorties numériques (temps, échantillons) sont celles du présent carnet ; la pédagogie est inchangée.

***

## 1. Introduction (~5 min)

### De la coloration de sommets à celle d'arêtes

Dans App-2, nous cherchions à colorier des **sommets** tels que deux sommets adjacents diffèrent. La **coloration d'arêtes** est le problème dual : colorier les **arêtes** d'un graphe telles que deux arêtes partageant un sommet reçoivent des couleurs distinctes. Le nombre minimal de couleurs est l'**indice chromatique** $\chi'(G)$.

Le théorème de **Vizing (1964)** encadre étroitement cet indice :

$$\Delta(G) \le \chi'(G) \le \Delta(G) + 1$$

où $\Delta(G)$ est le degré maximal. Les graphes atteignant la borne inférieure sont de **classe 1**, les autres de **classe 2**.

### Le cœur du problème : les graphes cubiques sans pont

Un graphe est **cubique** (ou 3-régulier) si tous ses sommets ont degré 3 ; il est **sans pont** si aucune arête n'est un pont (isthme). Pour un graphe cubique, $\chi'(G) \in \{3, 4\}$ : une 3-coloration d'arêtes d'un cubique revient à **partitionner ses arêtes en 3 couplages parfaits** — chaque couleur couvre chaque sommet exactement une fois.

La question « quel graphe cubique sans pont est de classe 1 ? » est structurellement difficile :

- **Conjecture de Tutte (1966)** : tout graphe cubique **planaire** sans pont est 3-arête-colorable — équivalente au théorème des quatre couleurs (résolue par conséquent en 1976).
- Un **snark** est un graphe cubique sans pont non 3-arête-colorable. Le plus petit est le **graphe de Petersen** (10 sommets).
- **Théorème (arXiv 2608.22870, 2026)** : tout graphe cubique **sans pont et apex** — un graphe dont la suppression d'**un seul sommet** le rend planaire — est 3-arête-colorable. Ce résultat étend la conjecture de Tutte au-delà du planaire, avec une preuve assistée par ordinateur et un algorithme constructif en $O(n^2)$.

Dans ce notebook, nous **vérifions empiriquement** ce théorème avec le solveur Z3. Le solveur peut fournir une coloration (statut `SAT`), prouver l'impossibilité (statut `UNSAT`), ou être interrompu par la limite de temps avant de conclure (statut indéterminé) ; ces trois situations restent explicitement distinctes.

In [2]:
Console.WriteLine("Z3 (Microsoft.Z3 5.1.0) pret");
Console.WriteLine("SEED = " + SEED);


Z3 (Microsoft.Z3 5.1.0) pret


SEED = 42


***

## 2. Définitions et familles témoins (~8 min)

Quatre propriétés structurelles vont orchestrer tout le notebook :

| Propriété | Définition | Vérification |
|---|---|---|
| **cubique** | tous les sommets ont degré 3 | parcours des degrés |
| **sans pont** | aucune arête dont la suppression déconnecte | parcours lowlink (Tarjan) |
| **planaire** | dessinable sans croisement | test gauche-droite (de Fraysseix–Rosenstiehl) |
| **apex** | il existe un sommet $v$ tel que $G - v$ soit planaire | test de planarité après suppression, pour chaque $v$ |

Remarquez l'emboîtement : **planaire $\Rightarrow$ apex** (supprimer un sommet d'un graphe planaire laisse un graphe planaire). L'intérêt du théorème de 2026 est précisément la zone **apex mais non planaire**, invisible pour Tutte-1966.

Nous assemblons nos familles témoins : $K_4$ et le prisme triangulaire (planaires triviaux), le graphe de Petersen (le snark minimal), et un graphe cubique **avec pont** construit à la main pour la section 4.

In [3]:
// Definitions structurelles (types du carnet)
class Graphe
{
    public List<List<int>> Adj { get; }
    public List<string> Noms { get; }

    public Graphe(List<string> noms)
    {
        Noms = noms;
        Adj = Enumerable.Range(0, noms.Count).Select(_ => new List<int>()).ToList();
    }

    public int N() => Adj.Count;
    public int M() => Adj.Select(v => v.Count).Sum() / 2;

    public void Ajoute(int u, int v)
    {
        if (u != v && !Adj[u].Contains(v)) { Adj[u].Add(v); Adj[v].Add(u); }
    }

    public void RetireArete(int u, int v) { Adj[u].Remove(v); Adj[v].Remove(u); }

    public Graphe Supprime(int v)
    {
        int[] nouveaux = new int[N()];
        for (int i = 0; i < N(); i++) nouveaux[i] = -1;
        var noms = new List<string>();
        for (int i = 0; i < N(); i++)
            if (i != v) { nouveaux[i] = noms.Count; noms.Add(Noms[i]); }
        var h = new Graphe(noms);
        for (int u = 0; u < N(); u++)
        {
            if (u == v) continue;
            foreach (int w in Adj[u])
                if (w != v) h.Ajoute(nouveaux[u], nouveaux[w]);
        }
        return h;
    }

    public List<(int, int)> Aretes()
    {
        var vs = new List<(int, int)>();
        for (int u = 0; u < N(); u++)
            foreach (int v in Adj[u])
                if (v > u) vs.Add((u, v));
        return vs;
    }
}

class Description
{
    public string Nom; public int N; public int M;
    public bool Cubique; public bool SansPontB; public bool Planaire; public bool Apex;
}



(52,49): warning CS0649: Le champ 'Description.M' n'est jamais assigné et aura toujours sa valeur par défaut 0

(53,61): warning CS0649: Le champ 'Description.Planaire' n'est jamais assigné et aura toujours sa valeur par défaut false

(53,17): warning CS0649: Le champ 'Description.Cubique' n'est jamais assigné et aura toujours sa valeur par défaut false

(52,19): warning CS0649: Le champ 'Description.Nom' n'est jamais assigné et aura toujours sa valeur par défaut null

(53,83): warning CS0649: Le champ 'Description.Apex' n'est jamais assigné et aura toujours sa valeur par défaut false

(53,38): warning CS0649: Le champ 'Description.SansPontB' n'est jamais assigné et aura toujours sa valeur par défaut false

(52,35): warning CS0649: Le champ 'Description.N' n'est jamais assigné et aura toujours sa valeur par défaut 0



In [4]:
// Test de planarite gauche-droite (de Fraysseix-Rosenstiehl), portage fidele de
// la partie test de networkx.algorithms.planarity (sans construction d'embedding).
struct Intervalle
{
    public int? Bas; public int? Haut;
    public bool EstVide() => !Bas.HasValue && !Haut.HasValue;
}

struct PaireConflit
{
    public Intervalle Gauche; public Intervalle Droite;
    public void Echange() { var t = Gauche; Gauche = Droite; Droite = t; }
}

class EtatLr
{
    public const int NON = int.MaxValue;
    readonly Graphe _g;
    readonly int n;
    readonly int[] prof;
    readonly int[] bas;
    readonly int[] bas2;
    readonly int[] nid;
    readonly int[] areteParent;
    readonly List<int>[] dg;
    readonly bool[] oriente;
    readonly List<int>[] ordonne;
    readonly Stack<PaireConflit> pile = new Stack<PaireConflit>();
    readonly int[] fondPile;
    readonly int[] areteBas;
    readonly int[] renvoi;
    readonly int[] cote;

    EtatLr(Graphe g)
    {
        _g = g; n = g.N();
        prof = new int[n]; for (int i = 0; i < n; i++) prof[i] = NON;
        bas = new int[n * n]; bas2 = new int[n * n]; nid = new int[n * n];
        areteParent = new int[n]; for (int i = 0; i < n; i++) areteParent[i] = NON;
        dg = new List<int>[n]; for (int i = 0; i < n; i++) dg[i] = new List<int>();
        oriente = new bool[n * n];
        ordonne = new List<int>[n]; for (int i = 0; i < n; i++) ordonne[i] = new List<int>();
        fondPile = new int[n * n];
        areteBas = new int[n * n]; for (int i = 0; i < n * n; i++) areteBas[i] = NON;
        renvoi = new int[n * n]; for (int i = 0; i < n * n; i++) renvoi[i] = NON;
        cote = new int[n * n]; for (int i = 0; i < n * n; i++) cote[i] = 1;
    }

    public static EtatLr Nouveau(Graphe g) => new EtatLr(g);

    public bool Executer()
    {
        for (int u = 0; u < n; u++)
            if (prof[u] == NON) { prof[u] = 0; Dfs1(u); }
        for (int u = 0; u < n; u++)
        {
            var ordre = new List<int>(dg[u]);
            ordre.Sort((w1, w2) => nid[u * n + w1].CompareTo(nid[u * n + w2]));
            ordonne[u] = ordre;
        }
        for (int u = 0; u < n; u++)
            if (areteParent[u] == NON && !Dfs2(u)) return false;
        return true;
    }

    void Dfs1(int u)
    {
        int e = areteParent[u];
        foreach (int w in _g.Adj[u])
        {
            int uw = u * n + w;
            int wu = w * n + u;
            if (oriente[uw] || oriente[wu]) continue;
            dg[u].Add(w);
            oriente[uw] = true;
            bas[uw] = prof[u];
            bas2[uw] = prof[u];
            if (prof[w] == NON)
            {
                areteParent[w] = uw;
                prof[w] = prof[u] + 1;
                Dfs1(w);
            }
            else
            {
                bas[uw] = prof[w];
            }
            nid[uw] = 2 * bas[uw];
            if (bas2[uw] < prof[u]) nid[uw] += 1;
            if (e != NON)
            {
                if (bas[uw] < bas[e])
                {
                    bas2[e] = Math.Min(bas2[e], bas2[uw]);
                    bas[e] = bas[uw];
                }
                else if (bas[uw] > bas[e])
                {
                    bas2[e] = Math.Min(bas2[e], bas2[uw]);
                }
                else
                {
                    bas2[e] = Math.Min(bas2[e], bas2[uw]);
                }
            }
        }
    }

    bool Conflit(Intervalle iv, int b) =>
        !iv.EstVide() && iv.Haut.HasValue && bas[iv.Haut.Value] > bas[b];

    bool Dfs2(int u)
    {
        int e = areteParent[u];
        foreach (int w in ordonne[u])
        {
            int ei = u * n + w;
            fondPile[ei] = pile.Count;
            if (ei == areteParent[w])
            {
                if (!Dfs2(w)) return false;
            }
            else
            {
                areteBas[ei] = ei;
                pile.Push(new PaireConflit
                {
                    Gauche = new Intervalle(),
                    Droite = new Intervalle { Bas = ei, Haut = ei }
                });
            }
            if (bas[ei] < prof[u])
            {
                if (w == ordonne[u][0])
                {
                    areteBas[e] = areteBas[ei];
                }
                else
                {
                    if (!AjouteContraintes(ei, e)) return false;
                }
            }
        }
        if (e != NON) RetireAretesRetour(e);
        return true;
    }

    bool AjouteContraintes(int ei, int e)
    {
        var p = new PaireConflit { Gauche = new Intervalle(), Droite = new Intervalle() };
        while (true)
        {
            if (pile.Count == 0) return false;
            var q = pile.Pop();
            if (!q.Gauche.EstVide()) q.Echange();
            if (!q.Gauche.EstVide()) return false;
            if (!q.Droite.Bas.HasValue) return false;
            int qbas = q.Droite.Bas.Value;
            if (bas[qbas] > bas[e])
            {
                if (p.Droite.EstVide())
                {
                    p.Droite = q.Droite;
                }
                else
                {
                    if (!p.Droite.Haut.HasValue || !q.Droite.Haut.HasValue) return false;
                    renvoi[p.Droite.Haut.Value] = q.Droite.Haut.Value;
                }
                p.Droite.Bas = q.Droite.Bas;
            }
            else
            {
                renvoi[qbas] = areteBas[e];
            }
            if (pile.Count == fondPile[ei]) break;
        }
        while (true)
        {
            if (pile.Count == 0) break;
            var top = pile.Peek();
            if (!Conflit(top.Gauche, ei) && !Conflit(top.Droite, ei)) break;
            var q = pile.Pop();
            if (Conflit(q.Droite, ei)) q.Echange();
            if (Conflit(q.Droite, ei)) return false;
            if (p.Droite.Bas.HasValue && q.Droite.Haut.HasValue)
                renvoi[p.Droite.Bas.Value] = q.Droite.Haut.Value;
            if (q.Droite.Bas.HasValue) p.Droite.Bas = q.Droite.Bas;
            if (p.Gauche.EstVide())
            {
                p.Gauche = q.Gauche;
            }
            else if (p.Gauche.Bas.HasValue && q.Gauche.Haut.HasValue)
            {
                renvoi[p.Gauche.Bas.Value] = q.Gauche.Haut.Value;
            }
            p.Gauche.Bas = q.Gauche.Bas;
        }
        if (!(p.Gauche.EstVide() && p.Droite.EstVide()))
            pile.Push(p);
        return true;
    }

    void RetireAretesRetour(int e)
    {
        int u = e / n;
        while (true)
        {
            if (pile.Count == 0) break;
            var top = pile.Peek();
            int PlusBas(Intervalle iv) => iv.Bas.HasValue ? bas[iv.Bas.Value] : NON;
            int plusBasPaire;
            if (top.Gauche.EstVide()) plusBasPaire = PlusBas(top.Droite);
            else if (top.Droite.EstVide()) plusBasPaire = PlusBas(top.Gauche);
            else plusBasPaire = Math.Min(PlusBas(top.Gauche), PlusBas(top.Droite));
            if (plusBasPaire == prof[u])
            {
                var pp = pile.Pop();
                if (pp.Gauche.Bas.HasValue) cote[pp.Gauche.Bas.Value] = -1;
            }
            else break;
        }
        if (pile.Count > 0)
        {
            var p = pile.Pop();
            while (p.Gauche.Haut.HasValue)
            {
                int h = p.Gauche.Haut.Value;
                if (h % n == u)
                {
                    int nxt = renvoi[h];
                    p.Gauche.Haut = nxt == NON ? (int?)null : nxt;
                }
                else break;
            }
            if (!p.Gauche.Haut.HasValue && p.Gauche.Bas.HasValue)
            {
                int x = p.Gauche.Bas.Value;
                renvoi[x] = p.Droite.Bas ?? NON;
                cote[x] = -1;
                p.Gauche.Bas = null;
            }
            while (p.Droite.Haut.HasValue)
            {
                int h = p.Droite.Haut.Value;
                if (h % n == u)
                {
                    int nxt = renvoi[h];
                    p.Droite.Haut = nxt == NON ? (int?)null : nxt;
                }
                else break;
            }
            if (!p.Droite.Haut.HasValue && p.Droite.Bas.HasValue)
            {
                int x = p.Droite.Bas.Value;
                renvoi[x] = p.Gauche.Bas ?? NON;
                cote[x] = -1;
                p.Droite.Bas = null;
            }
            pile.Push(p);
        }
        if (bas[e] < prof[u] && pile.Count > 0)
        {
            var top = pile.Peek();
            int? hl = top.Gauche.Haut;
            int? hr = top.Droite.Haut;
            renvoi[e] = (hl, hr) switch
            {
                (int h, null) => h,
                (int h, int r) when bas[h] > bas[r] => h,
                (int, int) => hr.Value,
                (null, int r) => r,
                (null, null) => NON,
            };
        }
    }
}


In [5]:
// Analyse structurelle (cubique / ponts / planarite / apex / isomorphisme)
class Analyse
{
    public static bool EstCubique(Graphe g) =>
        g.N() > 0 && g.Adj.All(vois => vois.Count == 3);

    // Ponts par parcours profond lowlink (Tarjan), iteratif.
    public static List<(int, int)> Ponts(Graphe g)
    {
        int n = g.N();
        var visites = new bool[n];
        var resultat = new List<(int, int)>();
        for (int depart = 0; depart < n; depart++)
        {
            if (visites[depart]) continue;
            visites[depart] = true;
            var prof = new int[n]; Array.Fill(prof, int.MaxValue);
            var bas = new int[n]; Array.Fill(bas, int.MaxValue);
            int horloge = 0;
            prof[depart] = 0; bas[depart] = 0;
            var pile = new Stack<(int u, int i, int pere)>();
            pile.Push((depart, 0, int.MaxValue));
            while (pile.Count > 0)
            {
                var (u, i, pere) = pile.Peek();
                if (i < g.Adj[u].Count)
                {
                    pile.Pop();
                    pile.Push((u, i + 1, pere));
                    int w = g.Adj[u][i];
                    if (w == pere) continue;
                    if (visites[w])
                    {
                        bas[u] = Math.Min(bas[u], prof[w]);
                    }
                    else
                    {
                        horloge++;
                        prof[w] = horloge; bas[w] = horloge;
                        visites[w] = true;
                        pile.Push((w, 0, u));
                    }
                }
                else
                {
                    pile.Pop();
                    if (pile.Count > 0)
                    {
                        var (parent, _, _) = pile.Peek();
                        bas[parent] = Math.Min(bas[parent], bas[u]);
                        if (bas[u] > prof[parent]) resultat.Add((parent, u));
                    }
                }
            }
        }
        return resultat;
    }

    public static bool SansPont(Graphe g) => Ponts(g).Count == 0;

    public static List<List<int>> ComposantesConnexes(Graphe g)
    {
        int n = g.N();
        var vus = new bool[n];
        var comps = new List<List<int>>();
        for (int depart = 0; depart < n; depart++)
        {
            if (vus[depart]) continue;
            vus[depart] = true;
            var pile = new Stack<int>(); pile.Push(depart);
            var comp = new List<int>();
            while (pile.Count > 0)
            {
                int u = pile.Pop();
                comp.Add(u);
                foreach (int w in g.Adj[u])
                    if (!vus[w]) { vus[w] = true; pile.Push(w); }
            }
            comp.Sort();
            comps.Add(comp);
        }
        return comps;
    }

    public static bool EstPlanaire(Graphe g)
    {
        int n = g.N();
        if (n <= 2) return true;
        if (g.M() > 3 * n - 6) return false;
        return EtatLr.Nouveau(g).Executer();
    }

    public static bool EstApex(Graphe g)
    {
        for (int v = 0; v < g.N(); v++)
            if (EstPlanaire(g.Supprime(v))) return true;
        return false;
    }

    public static Description Decrire(string nom, Graphe g) => new Description
    {
        Nom = nom, N = g.N(), M = g.M(),
        Cubique = EstCubique(g),
        SansPontB = SansPont(g),
        Planaire = EstPlanaire(g),
        Apex = EstApex(g),
    };

    // Isomorphisme par backtracking : bijection sommets preservant l'adjacence.
    public static bool EstIsomorphe(Graphe g1, Graphe g2)
    {
        int n = g1.N();
        if (n != g2.N() || g1.M() != g2.M()) return false;
        var correspondance = new int[n];
        var utilise = new bool[n];
        bool Adjacents(int a, int b, Graphe g) => g.Adj[a].Contains(b);
        bool Essaie(int i)
        {
            if (i == n) return true;
            for (int c = 0; c < n; c++)
            {
                if (utilise[c] || g1.Adj[i].Count != g2.Adj[c].Count) continue;
                bool coherent = true;
                for (int j = 0; j < i; j++)
                    if (Adjacents(i, j, g1) != Adjacents(c, correspondance[j], g2))
                    { coherent = false; break; }
                if (!coherent) continue;
                utilise[c] = true;
                correspondance[i] = c;
                if (Essaie(i + 1)) return true;
                utilise[c] = false;
            }
            return false;
        }
        return Essaie(0);
    }
}

// Constructeurs de graphes
class Graphes
{
    static List<string> Noms(int k) => Enumerable.Range(0, k).Select(i => i.ToString()).ToList();

    public static Graphe Complet(int k)
    {
        var g = new Graphe(Noms(k));
        for (int i = 0; i < k; i++)
            for (int j = i + 1; j < k; j++) g.Ajoute(i, j);
        return g;
    }

    public static Graphe PrismeTriangulaire()
    {
        var g = new Graphe(Noms(6));
        for (int i = 0; i < 3; i++)
        {
            g.Ajoute(i, (i + 1) % 3);
            g.Ajoute(i + 3, (i + 1) % 3 + 3);
            g.Ajoute(i, i + 3);
        }
        return g;
    }

    public static Graphe Petersen()
    {
        var g = new Graphe(Noms(10));
        for (int i = 0; i < 5; i++)
        {
            g.Ajoute(i, (i + 1) % 5);
            g.Ajoute(i + 5, (i + 2) % 5 + 5);
            g.Ajoute(i, i + 5);
        }
        return g;
    }

    // Deux copies de K4 moins une arete, recollees par un sommet de chaque cote,
    // relies par un unique pont. Cubique, avec exactement un pont.
    public static Graphe CubiqueAPont()
    {
        var noms = new List<string>();
        foreach (string tag in new[] { "a", "b" })
            for (int i = 0; i < 4; i++) noms.Add(tag + i);
        noms.Add("wa"); noms.Add("wb");
        var g = new Graphe(noms);
        foreach (int tag in new[] { 0, 4 })
            for (int i = 0; i < 4; i++)
                for (int j = i + 1; j < 4; j++)
                    if ((i, j) != (2, 3)) g.Ajoute(tag + i, tag + j);
        int wa = 8, wb = 9;
        g.Ajoute(wa, 2); g.Ajoute(wa, 3);
        g.Ajoute(wb, 6); g.Ajoute(wb, 7);
        g.Ajoute(wa, wb);
        return g;
    }

    // Echelle de Mobius M_n : cycle de n sommets + barreaux i -- (i + n/2).
    public static Graphe EchelleMobius(int n)
    {
        var g = new Graphe(Noms(n));
        for (int i = 0; i < n; i++) g.Ajoute(i, (i + 1) % n);
        for (int i = 0; i < n / 2; i++) g.Ajoute(i, (i + n / 2) % n);
        return g;
    }

    public static Graphe CompletBiparti(int p, int q)
    {
        var g = new Graphe(Noms(p + q));
        for (int i = 0; i < p; i++)
            for (int j = p; j < p + q; j++) g.Ajoute(i, j);
        return g;
    }

    public static Graphe Hypercube3()
    {
        var noms = Enumerable.Range(0, 8).Select(i => i.ToString("000")).ToList();
        var g = new Graphe(noms);
        for (int u = 0; u < 8; u++)
            for (int b = 0; b < 3; b++) g.Ajoute(u, u ^ (1 << b));
        return g;
    }

    // Graphe de Heawood : 7 points, 7 droites de l'incidence de Fano ;
    // la droite i porte les points i, i+1 et i+3 (mod 7).
    public static Graphe Heawood()
    {
        var noms = new List<string>();
        for (int i = 0; i < 7; i++) noms.Add("p" + i);
        for (int i = 0; i < 7; i++) noms.Add("d" + i);
        var g = new Graphe(noms);
        for (int i = 0; i < 7; i++)
        {
            g.Ajoute(i, 7 + i);
            g.Ajoute(i, 7 + (i + 1) % 7);
            g.Ajoute(i, 7 + (i + 3) % 7);
        }
        return g;
    }

    // Modele d'appariement (pairing model), portage du generateur
    // networkx.random_regular_graph.
    public static Graphe RegulierAleatoire(int d, int n, Random rng)
    {
        bool Contient(List<(int, int)> aretes, int s1, int s2) =>
            aretes.Any(ab => ab.Item1 == s1 && ab.Item2 == s2);
        while (true)
        {
            var aretes = new List<(int, int)>();
            var stubs = new List<int>();
            for (int sommet = 0; sommet < n; sommet++)
                for (int k = 0; k < d; k++) stubs.Add(sommet);
            bool abandon = false;
            int garde = 0;
            while (stubs.Count > 0)
            {
                // Garde anti-boucle : un appariement qui ne converge plus
                // (stubs residuels ne pouvant former que des aretes paralleles
                // ou des boucles) est abandonne et relance, au lieu de tourner
                // indefiniment sur les memes stubs.
                if (++garde > 1000) { abandon = true; break; }
                for (int i = stubs.Count - 1; i > 0; i--)
                {
                    int j = rng.Next(i + 1);
                    (stubs[i], stubs[j]) = (stubs[j], stubs[i]);
                }
                var kept = new List<int>();
                for (int i = 0; i + 1 < stubs.Count; i += 2)
                {
                    int s1 = stubs[i], s2 = stubs[i + 1];
                    if (s1 > s2) (s1, s2) = (s2, s1);
                    if (s1 != s2 && !Contient(aretes, s1, s2))
                    {
                        aretes.Add((s1, s2));
                    }
                    else
                    {
                        kept.Add(s1); kept.Add(s2);
                    }
                }
                stubs = kept;
            }
            if (abandon) continue;
            var g = new Graphe(Noms(n));
            foreach (var (a, b) in aretes) g.Ajoute(a, b);
            if (g.Adj.All(v => v.Count == d)) return g;
        }
    }
}

// Solveur Z3 de la k-coloration d'aretes
class Resolveur
{
    public static (List<int> Coloration, string Statut) ColorationAretesZ3(Graphe g, int k, double timeLimitS)
    {
        var aretes = g.Aretes();
        int na = aretes.Count;
        var ctx = new Context();
        var xv = new BoolExpr[na, k];
        for (int e = 0; e < na; e++)
            for (int c = 0; c < k; c++)
                xv[e, c] = ctx.MkBoolConst($"x{e}_{c}");
        var s = ctx.MkSolver();
        // Z3 4.13.0 : le timeout se fixe sur le Solver via Set("timeout", ms) ;
        // 0 ms force une interruption immediate (statut UNKNOWN).
        s.Set("timeout", timeLimitS > 0 ? (uint)(timeLimitS * 1000) : 1u);
        for (int e = 0; e < na; e++)
        {
            var xs = new List<BoolExpr>();
            for (int c = 0; c < k; c++) xs.Add(xv[e, c]);
            s.Assert(ctx.MkAtMost(xs, 1u));   // au plus une couleur par arete
            s.Assert(ctx.MkAtLeast(xs, 1u));   // + au moins une = exactement une
        }
        for (int vv = 0; vv < g.N(); vv++)
            for (int c = 0; c < k; c++)
            {
                var list = new List<BoolExpr>();
                for (int e = 0; e < na; e++)
                    if (aretes[e].Item1 == vv || aretes[e].Item2 == vv) list.Add(xv[e, c]);
                if (list.Count > 0) s.Assert(ctx.MkAtMost(list, 1u));
            }
        var r = s.Check();
        if (r == Status.SATISFIABLE)
        {
            var m = s.Model;
            var trueLit = ctx.MkTrue();
            var coloration = new List<int>(Enumerable.Repeat(0, na));
            for (int e = 0; e < na; e++)
                for (int c = 0; c < k; c++)
                    if (m.Evaluate(xv[e, c], true).Equals(trueLit)) coloration[e] = c;
            return (coloration, "SAT");
        }
        if (r == Status.UNSATISFIABLE) return (null, "UNSAT");
        return (null, "UNKNOWN");
    }

    public static string VerdictTexte(List<int> coloration, string statut)
    {
        if (coloration != null) return "3-colorable";
        if (statut == "UNSAT") return "INFEASIBLE (chi' >= 4)";
        return $"INDETERMINE ({statut})";
    }
}

// Affichage textuel
class Affichage
{
    public static string V(bool b) => b ? "True" : "False";

    public static void TableTemoins(List<Description> descs)
    {
        Console.WriteLine(
            "{0,-22}{1,3}{2,4}  {3,-8}{4,-10}{5,-9}{6}",
            "graphe", "n", "m", "cubique", "sans_pont", "planaire", "apex");
        foreach (var d in descs)
            Console.WriteLine(
                "{0,-22}{1,3}{2,4}  {3,-8}{4,-10}{5,-9}{6}",
                d.Nom, d.N, d.M, V(d.Cubique), V(d.SansPontB), V(d.Planaire), V(d.Apex));
    }

    public static void AfficheAdjacences(Graphe g, string titre, (int, int)? marque)
    {
        Console.WriteLine(titre);
        for (int u = 0; u < g.N(); u++)
        {
            var voisins = g.Adj[u].Select(w =>
            {
                bool estMarque = false;
                if (marque.HasValue)
                {
                    int ma = marque.Value.Item1, mb = marque.Value.Item2;
                    estMarque = (ma == u && mb == w) || (ma == w && mb == u);
                }
                return estMarque ? $"[{g.Noms[w]}]" : g.Noms[w];
            }).ToList();
            Console.WriteLine($"  {g.Noms[u],4} : {string.Join(" ", voisins)}");
        }
    }

    public static void AfficheColoration(Graphe g, string titre, int k, List<int> coloration)
    {
        var aretes = g.Aretes();
        Console.WriteLine(titre);
        for (int c = 0; c < k; c++)
        {
            var aretesC = aretes
                .Select((ab, i) => (ab, i))
                .Where(t => coloration[t.i] == c)
                .Select(t => $"{g.Noms[t.ab.Item1]}-{g.Noms[t.ab.Item2]}")
                .ToList();
            Console.WriteLine($"  couleur {c} : {string.Join(", ", aretesC)}");
        }
    }
}


In [6]:
// Familles temoins (top-level : partage par les cellules suivantes)
var TEMOINS = new (string, Graphe)[]
{
    ("K4", Graphes.Complet(4)),
    ("prisme triangulaire", Graphes.PrismeTriangulaire()),
    ("Petersen", Graphes.Petersen()),
    ("cubique a pont", Graphes.CubiqueAPont()),
};
{
    var descs = TEMOINS.Select(t => Analyse.Decrire(t.Item1, t.Item2)).ToList();
    Affichage.TableTemoins(descs);
}


graphe                  n   m  cubique sans_pont planaire apex


K4                      4   6  True    True      True     True


prisme triangulaire     6   9  True    True      True     True


Petersen               10  15  True    True      False    False


cubique a pont         10  15  True    False     True     True


### Interprétation : lecture du tableau

Quatre témoins, quatre cas : $K_4$ et le prisme triangulaire (planaires **et** apex), le graphe de Petersen (non planaire, non apex) et le cubique à pont (planaire, apex, **avec** un pont). L'emboîtement **planaire $\Rightarrow$ apex** s'y lit directement : les deux graphes planaires sont apex, et le graphe à pont l'est aussi — mais il garde son pont, qui le sort du périmètre des deux théorèmes.

C'est la **combinaison** des propriétés (cubique, sans-pont, apex) qui va prédire la colorabilité dans la section 3 : le solveur Z3 va confirmer, graphe par graphe, ce que la structure annonce.

In [7]:
// Temoins de la section 2 (top-level : PETERSEN / G_PONT reutilises plus bas)
var PETERSEN = TEMOINS[2].Item2;
var G_PONT = TEMOINS[3].Item2;
{
    Affichage.AfficheAdjacences(PETERSEN, "Graphe de Petersen (10 sommets, 15 aretes) :", null);
    Console.WriteLine();
    var pb = Analyse.Ponts(G_PONT).FirstOrDefault();
    Affichage.AfficheAdjacences(G_PONT, "Cubique a pont -- le pont entre crochets :", pb);
}


Graphe de Petersen (10 sommets, 15 aretes) :


     0 : 1 5 4


     1 : 0 2 6


     2 : 1 3 7


     3 : 2 4 8


     4 : 3 0 9


... (12 outputs left out)

    b2 : b0 b1 wb


    b3 : b0 b1 wb


    wa : a2 a3 [wb]


    wb : b2 b3 [wa]


La liste d'adjacence fait toute la pédagogie du graphe de Petersen : chaque sommet $i$ y a **exactly** trois voisins — le cycle externe ($i \pm 1$), le centre de l'étoile ($i+2$) et le rayonnement ($i+5$).

Le graphe à pont se lit encore plus vite : l'arête $[wa]-wb$ est le **seul** isthme ; la liste se coupe en deux moitiés (les sommets $a_\ast$ d'un côté, $b_\ast$ de l'autre) autour de ce pont — l'asymétrie que l'on exploite en section 4.

> La version textuelle remplace le dessin matplotlib de la version Python : en C# headless, le calcul NuGet fonctionne mais pas l'affichage graphique. La structure se lit dans les listes d'adjacence, qui sont en définitive la représentation exacte que le solveur Z3 consomme.

***

## 3. Formulation CSP et solveur Z3 (~10 min)

### La 3-coloration comme contrainte de couplage parfait

Pour un graphe cubique $G$ à $n$ sommets, une 3-coloration d'arêtes est équivalente à une **partition des $3n/2$ arêtes en 3 couplages parfaits**. Le solveur Z3 est appelé avec $k$ comme paramètre : la même instance décide alors si $\chi'(G) \le k$, ce qui donne à la fois le test du théorème (pour $k=3$) et la mesure de l'indice chromatique exact (en remontant $k$).

Les contraintes sont exactement celles de la version Python, écrites ici avec l'API `Microsoft.Z3` :

- **AMO** : pour chaque arête, au plus une couleur ;
- **ALO** : pour chaque arête, au moins une couleur (AMO + ALO = exactement une) ;
- **per-sommet** : pour chaque sommet et chaque couleur, au plus une arête incidente de cette couleur (c'est le couplage parfait).

La différence d'outillage avec la version Python est d'essence : CP-SAT est un solveur *dédié* aux problèmes d'optimisation combinatoire entiers, tandis que Z3 est un *SMT* généraliste — on lui exprime ici une théorie booléenne pure, ce qui rend la formulation plus directe, sans la spécialisation du CP-SAT.

In [8]:
{
    foreach (var (nom, g) in TEMOINS)
    {
        var t0 = System.Diagnostics.Stopwatch.StartNew();
        var (coloration, statut) = Resolveur.ColorationAretesZ3(g, 3, 10.0);
        t0.Stop();
        Console.WriteLine("{0,-22} -> {1,-28} ({2,5:F1} ms)",
            nom, Resolveur.VerdictTexte(coloration, statut), t0.Elapsed.TotalMilliseconds);
    }
    var t4 = System.Diagnostics.Stopwatch.StartNew();
    var (col4, statut4) = Resolveur.ColorationAretesZ3(PETERSEN, 4, 10.0);
    t4.Stop();
    string etat4 = col4 != null ? "trouve" : "non trouve (" + statut4 + ")";
    Console.WriteLine("Petersen en 4 couleurs -> {0} ({1,5:F1} ms)", etat4, t4.Elapsed.TotalMilliseconds);

    // Le solveur Z3 renvoie trois statuts ; VerdictTexte mappe chacun en un texte lisible.
    // On affiche les trois mappings directement (sans solve) pour rester deterministe sur
    // toute machine : SAT (coloration trouvee), UNSAT (impossibilite prouvee),
    // UNKNOWN (interruption du solveur quand le budget de temps est depasse).
    // Sur les petits temoins ci-dessus, Z3 conclut avant toute interruption, d'ou
    // les deux premiers statuts dans la boucle ; la case UNKNOWN ne s'y manifeste pas.
    Console.WriteLine("cartographie des 3 statuts du solveur (VerdictTexte) :");
    Console.WriteLine("  SAT     -> " + Resolveur.VerdictTexte(new List<int> { 0, 1, 2 }, "SAT"));
    Console.WriteLine("  UNSAT   -> " + Resolveur.VerdictTexte(null, "UNSAT"));
    Console.WriteLine("  UNKNOWN -> " + Resolveur.VerdictTexte(null, "UNKNOWN"));
}


K4                     -> 3-colorable                  (117,3 ms)


prisme triangulaire    -> 3-colorable                  ( 38,1 ms)


Petersen               -> INFEASIBLE (chi' >= 4)       (  7,2 ms)


cubique a pont         -> INFEASIBLE (chi' >= 4)       (  7,9 ms)


Petersen en 4 couleurs -> trouve (  8,1 ms)


cartographie des 3 statuts du solveur (VerdictTexte) :


  SAT     -> 3-colorable


  UNSAT   -> INFEASIBLE (chi' >= 4)


  UNKNOWN -> INDETERMINE (UNKNOWN)


### Interprétation du solveur

Les quatre témoins donnent la **signature exacte** du théorème : $K_4$ et le prisme (planaires, apex) sont 3-colorables ; le Petersen (snark) et le cubique à pont (pont + apex) sont **INFEASIBLE** à $k=3$ — le solveur **prouve** l'impossibilité, il ne se contente pas de ne pas trouver.

Remarquez le **contraste** entre le Petersen et le cubique à pont : tous deux apex, tous deux INFEASIBLE, mais pour des raisons **différentes** — le Petersen l'est parce qu'il est un snark (obstacle intrinsèque), le cubique à pont l'est **à cause de son pont** (obstacle structurel, section 4). Le solveur ne fait pas la distinction ; c'est l'analyse structurelle qui la fait.

Le solveur Z3 renvoie en réalité **trois** statuts, pas deux : `SAT` (coloration trouvée), `UNSAT` (impossibilité prouvée) et `UNKNOWN` — le cas où le solveur est **interrompu** parce que le budget de temps est dépassé avant qu'il n'ait pu conclure. Sur les petits témoins de ce notebook, Z3 conclut toujours bien avant toute interruption (la preuve UNSAT du Petersen tient en quelques millisecondes), si bien que seuls `SAT` et `UNSAT` s'y manifestent ; la case `UNKNOWN` n'apparaît que sur des instances assez denses pour épuiser le budget. Le `VerdictTexte` du code mappe **les trois** vers un texte lisible, et la ligne de cartographie ci-dessous les affiche explicitement : la troisième case du tableau de vérité du solveur y reste visible, sans dépendre d'une interruption fortuite.

**Ce que C# apporte.** Comme le portage Rust, C# supprime le surcoût fixe d'appel du Python (import, construction de l'objet problème, sérialisation) : le temps résiduel mesuré ici est celui du **résolution Z3 elle-même**, très petit sur ces instances jouet. L'écart d'ordre de grandeur avec la version Python vient donc du harnais d'appel, pas de la difficulté du problème.

In [9]:
{
    // Temoins fixes : prisme (classe 1, chi'=3) puis Petersen (snark, chi'=4) via Z3.
    var prisme = TEMOINS[1].Item2;
    var (colP, statutP) = Resolveur.ColorationAretesZ3(prisme, 3, 10.0);
    if (colP == null) throw new Exception("coloration du prisme non trouvee : " + statutP);
    Affichage.AfficheColoration(prisme, "Prisme : 3-coloration d'aretes (chi' = 3) :", 3, colP);
    Console.WriteLine();

    var (col4b, statut4b) = Resolveur.ColorationAretesZ3(PETERSEN, 4, 10.0);
    if (col4b == null) throw new Exception("4-coloration de Petersen non trouvee : " + statut4b);
    Affichage.AfficheColoration(PETERSEN, "Petersen : il faut 4 couleurs (chi' = 4) -- snark :", 4, col4b);

    var aretesPrisme = prisme.Aretes();
    for (int c = 0; c < 3; c++)
    {
        var sommetsC = new List<int>();
        for (int i = 0; i < aretesPrisme.Count; i++)
            if (colP[i] == c) { sommetsC.Add(aretesPrisme[i].Item1); sommetsC.Add(aretesPrisme[i].Item2); }
        sommetsC.Sort();
        Console.WriteLine("prisme, couleur {0} : {1} aretes couvrant {2} sommets",
            c, sommetsC.Count / 2, string.Join(", ", sommetsC));
        if (sommetsC.Count != 6) throw new Exception("couleur ne couvre pas les 6 sommets");
    }
}


Prisme : 3-coloration d'aretes (chi' = 3) :


  couleur 0 : 0-3, 1-2, 4-5


  couleur 1 : 0-1, 2-5, 3-4


  couleur 2 : 0-2, 1-4, 3-5


Petersen : il faut 4 couleurs (chi' = 4) -- snark :


  couleur 0 : 0-5, 1-2, 3-4, 6-8, 7-9


... (2 outputs left out)

  couleur 3 : 0-4, 2-3


prisme, couleur 0 : 3 aretes couvrant 0, 1, 2, 3, 4, 5 sommets


prisme, couleur 1 : 3 aretes couvrant 0, 1, 2, 3, 4, 5 sommets


prisme, couleur 2 : 3 aretes couvrant 0, 1, 2, 3, 4, 5 sommets


### La coloration renvoyée : un couplage parfait par couleur

Sur le prisme, chaque couleur recouvre exactement un couplage parfait : 3 arêtes, les 6 sommets, aucun sommet touché deux fois. C'est la **décomposition** que le théorème promettait, et le code la **vérifie** (chaque couleur couvre bien les 6 sommets).

Sur le Petersen, il faut 4 couleurs : la 4ᵉ existe pour absorber les arêtes que les 3 premières ne peuvent pas couvrir ensemble — la signature du snark.

La forme textuelle est moins parlante qu'un graphe coloré, mais elle **prouve** la structure : on peut vérifier à la main que chaque couleur forme un couplage, sans dépendre d'un rendu graphique.

***

## 4. Le théorème sans-pont : parité et ponts (~8 min)

### Le pont force l'imparité

Le graphe cubique à pont est **planaire et apex** — il devrait donc être 3-colorable selon Tutte-1966 **et** selon apex-2026. Et pourtant il est **INFEASIBLE** à $k=3$. La contradiction apparente se résout en une ligne : le pont.

Supprimons l'arête pont $(wa, wb)$. Le graphe se coupe en deux composantes $A$ et $B$. Dans un cubique, chaque sommet de $A$ (sauf $wa$) a ses 3 arêtes internes ; $wa$ en a 2 internes + 1 vers le pont. Le compte d'arêtes internes de $A$ est donc :

$$3|A| - 1 = 2|E_A|$$

Le membre de gauche est **impair**, le membre de droit **pair** : c'est précisément l'incohérence que le notebook **mesure** en calculant le nombre d'arêtes internes de chaque composante après suppression du pont.

L'argument s'énonce plus simplement en termes de couplages : une 3-coloration d'un cubique partitionne ses arêtes en 3 couplages **parfaits**, et chaque couplage parfait traverse un pont un nombre **pair** de fois par composante. Un pont, qui ne peut être couvert que par une **seule** couleur (une seule arête), rompt la parité : il n'existe pas de 3-coloration.

Le solveur **confirme** la conséquence (INFEASIBLE à $k=3$), indépendamment de l'argument de parité : deux preuves, deux natures, même conclusion — c'est le **pont** qui est l'obstacle, pas la non-planarité ni le défaut d'apex.

In [10]:
{
    var listePonts = Analyse.Ponts(G_PONT);
    var nomsPonts = listePonts.Select(ab =>
        G_PONT.Noms[ab.Item1] + ", " + G_PONT.Noms[ab.Item2]).ToList();
    Console.WriteLine("ponts : [{0}]", string.Join(", ", nomsPonts));

    var (pa, pb2) = listePonts[0];
    var h = Graphes.CubiqueAPont();   // copie fraiche identique
    h.RetireArete(pa, pb2);
    int i = 0;
    foreach (var comp in Analyse.ComposantesConnexes(h))
    {
        int nbAretes = 0;
        foreach (int u in comp)
            foreach (int w in h.Adj[u])
                if (comp.Contains(w)) nbAretes++;
        nbAretes /= 2;
        string parite = comp.Count % 2 == 0 ? "pair" : "IMPAIR";
        Console.WriteLine("composante {0} : |A| = {1} (parite {2}), 3|A| = {3}, 2|E_A| + 1 = {4}",
            i, comp.Count, parite, 3 * comp.Count, 2 * nbAretes + 1);
        i++;
    }

    var (col3Pont, statutPont) = Resolveur.ColorationAretesZ3(G_PONT, 3, 10.0);
    Console.WriteLine("3-coloration du graphe a pont : {0}", Resolveur.VerdictTexte(col3Pont, statutPont));
}


ponts : [wa, wb]


composante 0 : |A| = 5 (parite IMPAIR), 3|A| = 15, 2|E_A| + 1 = 15


composante 1 : |A| = 5 (parite IMPAIR), 3|A| = 15, 2|E_A| + 1 = 15


3-coloration du graphe a pont : INFEASIBLE (chi' >= 4)


***

## 5. Le théorème apex : les échelles de Möbius (~8 min)

### La famille qui rend le théorème non-trivial

Les **échelles de Möbius** $M_{2n}$ (cycle de $2n$ sommets + barreaux opposés) sont la famille témoin du théorème apex-2026 :

- **non planaires** dès $M_8$ (elles contiennent une subdivision de $K_5$ ou de $K_{3,3}$) ;
- **apex** : supprimer un sommet quelconque du cycle laisse un graphe planaire (l'échelle ordinaire) ;
- **sans pont** : le cycle externe assure la 2-connexité.

Elles vivent donc **exactement** dans la zone « apex mais non planaire » que Tutte-1966 ne couvre pas — et le théorème 2026 les prédit 3-colorables. C'est le test le plus discriminant du notebook.

In [11]:
{
    // Echelles de Mobius M8/M10/M12 : 3-colorabilite + temps de resolution Z3.
    var MOBIUS = new (string, Graphe)[]
    {
        ("M8", Graphes.EchelleMobius(8)),
        ("M10", Graphes.EchelleMobius(10)),
        ("M12", Graphes.EchelleMobius(12)),
    };
    foreach (var (nom, g) in MOBIUS)
    {
        var t0 = System.Diagnostics.Stopwatch.StartNew();
        var (coloration, statut) = Resolveur.ColorationAretesZ3(g, 3, 10.0);
        t0.Stop();
        string verdict = coloration != null ? "oui" : "non (" + statut + ")";
        Console.WriteLine("{0,-5} 3-colorable = {1} ({2,5:F1} ms)", nom, verdict, t0.Elapsed.TotalMilliseconds);
    }
    Console.WriteLine();
    var descs = MOBIUS.Select(t => Analyse.Decrire(t.Item1, t.Item2)).ToList();
    Affichage.TableTemoins(descs);
}


M8    3-colorable = oui (  5,6 ms)


M10   3-colorable = oui (  5,6 ms)


M12   3-colorable = oui (  5,8 ms)


graphe                  n   m  cubique sans_pont planaire apex


M8                      8  12  True    True      False    True


M10                    10  15  True    True      False    True


M12                    12  18  True    True      False    True


Les trois échelles $M_8, M_{10}, M_{12}$ donnent la **signature complète** : **non planaires** (le test de planarité rend `False`), **apex** (`True`), et **3-colorables** (le solveur trouve). C'est le théorème apex-2026 en action, sur la famille qui le rend non-trivial.

Le tableau (planaire, apex) appliqué à la famille de Möbius fait apparaître l'emboîtement **non-planaire $\subset$ apex** que le théorème exploite : les trois échelles sont apex, aucune n'est planaire, et toutes sont 3-colorables.

### Echantillonnage : la prévision tient-elle sur 48 instances aléatoires ?

Le test le plus fort du théorème est un **test sur population** : générer des graphes cubiques **aléatoires** (modèle d'appariement), filtrer les **sans pont**, et vérifier que **aucun** d'eux n'est à la fois **apex** et **INFEASIBLE** à $k=3$.

C'est exactement la propriété que le théorème 2026 affirme : sur la population « cubique sans pont », le prédicat « apex » **implique** « 3-colorable ». Un seul contre-exemple apex+INFEASIBLE tuerait la prévision ; zéro contre-exemple sur 48 instances la **soutient** (sans la prouver — c'est un test empirique).

La version C# échantillonne 12 graphes de chaque taille $n \in \{10, 12, 14, 16\}$, soit 48 instances, avec le même modèle d'appariement que la version Python (le générateur `RegulierAleatoire` est un portage direct de `networkx.random_regular_graph`).

In [12]:
{
    // Echantillonnage de graphes cubiques aleatoires sans pont : apex / planaire / snark.
    var rng = new Random(SEED);
    var lignes = new List<(int n, bool pl, bool ap, string statut, bool c3, double ms)>();
    int nApex = 0, nNonApex = 0, nSnark = 0, nIndetermine = 0;
    foreach (int n in new[] { 10, 12, 14, 16 })
    {
        int trouves = 0, essais = 0;
        while (trouves < N_TRIES_PAR_TAILLE && essais < 200)
        {
            essais++;
            var graine = rng.Next();
            var g = Graphes.RegulierAleatoire(3, n, new Random(graine));
            if (!Analyse.SansPont(g)) continue;
            trouves++;
            var t0 = System.Diagnostics.Stopwatch.StartNew();
            var (col, statut) = Resolveur.ColorationAretesZ3(g, 3, 10.0);
            t0.Stop();
            bool ap = Analyse.EstApex(g);
            bool pl = Analyse.EstPlanaire(g);
            lignes.Add((n, pl, ap, statut, col != null, Math.Round(t0.Elapsed.TotalMilliseconds, 1)));
            if (ap) nApex++;
            if (!ap) nNonApex++;
            if (statut == "UNSAT") nSnark++;
            if (statut == "UNKNOWN") nIndetermine++;
        }
    }
    Console.WriteLine("instances sans pont echantillonnees : {0}", lignes.Count);
    Console.WriteLine("  apex              : {0}", nApex);
    Console.WriteLine("  non apex          : {0}", nNonApex);
    Console.WriteLine("  snarks (INFEASIBLE): {0}", nSnark);
    Console.WriteLine("  indetermines       : {0}", nIndetermine);
    Console.WriteLine("  apex ET INFEASIBLE : {0}   (attendu 0)",
        lignes.Count(l => l.ap && l.statut == "UNSAT"));

    var comptage = new SortedDictionary<(int, bool, bool, string, bool), int>();
    foreach (var l in lignes)
    {
        var key = (l.n, l.pl, l.ap, l.statut, l.c3);
        comptage[key] = comptage.GetValueOrDefault(key) + 1;
    }
    Console.WriteLine();
    Console.WriteLine("{0,3}  {1,9}  {2,5}  {3,8}  {4,11}  nb", "n", "planaire", "apex", "statut", "3-colorable");
    foreach (var (key, nb) in comptage)
    {
        Console.WriteLine("{0,3}  {1,9}  {2,5}  {3,8}  {4,11}  {5}",
            key.Item1, Affichage.V(key.Item2), Affichage.V(key.Item3), key.Item4, Affichage.V(key.Item5), nb);
    }
}


instances sans pont echantillonnees : 48


  apex              : 46


  non apex          : 2


  snarks (INFEASIBLE): 0


  indetermines       : 0


  apex ET INFEASIBLE : 0   (attendu 0)


... (7 outputs left out)

 14       True   True       SAT         True  2


 16      False  False       SAT         True  1


 16      False   True       SAT         True  9


 16       True   True       SAT         True  2


L'échantillonnage donne la **confirmation quantitative** du théorème : sur les graphes cubiques sans pont aléatoires, on compte les apex, les non-apex, les snarks (INFEASIBLE) — et surtout le croisement **apex ET INFEASIBLE**, qui doit être **zéro**.

Le tableau de distribution (par taille, planarité, apex, statut) fait apparaître la **structure** de la population : les apex sont une minorité, les snarks une minorité, et les deux ensembles sont **disjoints** sur l'échantillon. C'est la forme empirique du théorème.

Le décompte exact (combien d'apex, combien de snarks) varie d'un carnet à l'autre selon la graine et le générateur aléatoire — ce qui ne change rien au verdict : c'est la **disjonction** apex $\cap$ INFEASIBLE $= \emptyset$ qui porte la prévision, pas les marges. (Le carnet Rust obtient un partage apex/non-apex légèrement différent, 46/2, à graine égale : les flux aléatoires de C# et de Rust ne sont pas les mêmes — le verdict, lui, est identique.)

***

## 6. Exercices (~12 min)

Trois exercices progressifs, chacun ancré sur une notion que le notebook a démontrée :

1. **L'indice chromatique exact** ($\chi'$) — utiliser le solveur pour mesurer $\chi'(G)$, pas seulement tester $k=3$ ;
2. **La classification de Vizing** — distinguer classe 1 de classe 2 sur les graphes complets, et chercher l'obstacle structurel d'un snark ;
3. **L'apex au-delà du planaire** — explorer la famille de Möbius et sa réciproque.

Chaque exercice suit le même schéma : un **exemple résolu** (la méthode, en C#) puis un **stub** à compléter (la transposition à un cas nouveau). Les stubs sont **compilables** : ils affichent `Exercice a completer` et renvoient `null`, sans aucune erreur volontaire — le carnet s'exécute de bout en bout même exercices non complétés.

> **Attribution.** Les trois exemples guidés ci-dessous (indice chromatique, chasse au snark, exploration de Möbius) sont distillés des livraisons du **groupe 7** (TP1, IA2-Min1, EPF) — Yasmine Nehad, Adam Benlaharche et Solène Martinez (PR étudiante #25629) ; les stubs en sont la transposition à un cas nouveau, non résolu.

### Exercice 1 (résolu) : l'indice chromatique exact

**Énoncé** : écrire une fonction `IndiceChromatique(G)` qui renvoie $\chi'(G)$, en s'appuyant sur le solveur Z3.

**Méthode** : par Vizing, $\chi'(G) \in \{\Delta(G), \Delta(G)+1\}$. On teste donc d'abord $k = \Delta(G)$ : si le solveur répond `SAT`, c'est l'indice ; s'il répond `UNSAT`, on remonte à $k = \Delta(G)+1$ (qui doit alors être `SAT`, sinon on a une contradiction avec Vizing — ce que le code signale).

Quatre cas valident : $K_4$ ($\chi'=3$), le Petersen ($\chi'=4$), $K_{3,3}$ ($\chi'=3$, bien que non cubique) et le cubique à pont ($\chi'=4$, l'obstacle étant le pont).

In [13]:
{
    // Indice chromatique d'aretes par dichotomie Z3 : verifie Vizing sur les temoins.
    int IndiceChromatique(Graphe g)
    {
        int delta = g.Adj.Select(v => v.Count).Max();
        var (coloration, statut) = Resolveur.ColorationAretesZ3(g, delta, 10.0);
        if (coloration != null) return delta;
        if (statut != "UNSAT") throw new Exception("statut indetermine pour k = " + delta + " : " + statut);
        var (coloration2, statut2) = Resolveur.ColorationAretesZ3(g, delta + 1, 10.0);
        if (coloration2 == null) throw new Exception("contredit Vizing ?! statut = " + statut2);
        return delta + 1;
    }

    var K33 = Graphes.CompletBiparti(3, 3);
    var cas = new (string, Graphe, int)[]
    {
        ("K4", TEMOINS[0].Item2, 3),
        ("Petersen", TEMOINS[2].Item2, 4),
        ("K33", K33, 3),
        ("cubique a pont", TEMOINS[3].Item2, 4),
    };
    foreach (var (nom, g, attendu) in cas)
    {
        int chi = IndiceChromatique(g);
        Console.WriteLine("{0,-15} chi' = {1}  (attendu {2})  {3}",
            nom, chi, attendu, chi == attendu ? "OK" : "ERREUR");
        if (chi != attendu) throw new Exception("ERREUR");
    }
}


K4              chi' = 3  (attendu 3)  OK


Petersen        chi' = 4  (attendu 4)  OK


K33             chi' = 3  (attendu 3)  OK


cubique a pont  chi' = 4  (attendu 4)  OK


<null>

La fonction **réutilise** le solveur déjà écrit — c'est toute la puissance de la formulation CSP : mesurer $\chi'$ ne demande qu'une boucle sur les deux valeurs autorisées par Vizing, pas un nouvel algorithme.

Le cas du cubique à pont est le plus instructif : $\Delta=3$, le solveur répond `UNSAT` à $k=3$, et `SAT` à $k=4$ — l'indice est 4, **à cause du pont** (section 4), pas parce que le graphe serait un snark (il est même planaire et apex).

### Exercice 1 (à compléter) : les classes de Vizing des graphes complets

**Énoncé** : les graphes complets $K_n$ sont-ils de classe 1 ou de classe 2 ? Écrire `ClassesVizingComplets(nMax)` qui renvoie la liste des $n \le n_{\max}$ tels que $K_n$ soit de **classe 2** ($\chi'(K_n) = \Delta(K_n) + 1$).

**Indice** : dans $K_n$ avec $n$ **impair**, un couplage parfait couvre au plus $(n-1)$ sommets (un sommet reste de côté) ; $K_n$ compte $\sqrt{n(n-1)/2}$ arêtes. Borner $k$ par en-dessous fait apparaître la réponse.

**Attendu** : $K_n$ est de classe 2 exactement quand $n$ est **impair** — la liste attendue est donc `[3, 5, 7]` pour `nMax = 7`.

In [14]:
{
    List<int> ClassesVizingComplets(int nMax)
    {
        // TODO: classez les graphes complets K_n par classe de Vizing
        // Etape 1 : construisez K_n avec Graphes.Complet(n) et mesurez chi' (IndiceChromatique)
        // Etape 2 : comparez chi'(K_n) a la borne Delta(K_n) = n - 1
        // Etape 3 : renvoyez la liste des n tels que K_n est de classe 2
        // Indice : dans K_n avec n impair, un couplage couvre au plus (n - 1)/2 aretes par sommet
        Console.WriteLine("Exercice a completer");
        return null;
    }

    ClassesVizingComplets(7);

    // Tests attendus (decommentez apres implementation) :
    // var classe2 = ClassesVizingComplets(7);
    // Console.WriteLine("K_n de classe 2 pour n dans : {0}", string.Join(", ", classe2));
    // if (string.Join(",", classe2) != "3,5,7") throw new Exception("ERREUR");
}


Exercice a completer


<null>

### Exercice 2 (résolu) : la chasse au snark

**Énoncé** : automatiser la recherche d'un **snark** (cubique, sans pont, INFEASIBLE à $k=3$) parmi les graphes cubiques aléatoires de taille 10, puis vérifier qu'il est **isomorphe** au graphe de Petersen.

**Méthode** : générer des cubiques aléatoires de 10 sommets (modèle d'appariement), filtrer les sans pont, tester la 3-colorabilité ; le premier INFEASIBLE est un snark. Puis comparer sa structure à celle de Petersen par **isomorphisme** (backtracking sur les permutations de sommets, avec coupe par compatibilité des degrés).

In [15]:
{
    // Population de cubiques sans pont de 10 sommets : on y ajoute Petersen lui-meme
    // (garanti : au moins un snark dans la population). Chaque graphe INFEASIBLE doit
    // etre isomorphe a Petersen -- le seul snark d'ordre 10.
    var rng = new Random(SEED);
    var pop = new List<Graphe>();
    pop.Add(PETERSEN);
    int ajoutes = 0;
    for (int i = 0; i < 60 && ajoutes < 15; i++)
    {
        var g = Graphes.RegulierAleatoire(3, 10, new Random(rng.Next()));
        if (Analyse.SansPont(g)) { pop.Add(g); ajoutes++; }
    }
    Console.WriteLine("population : {0} graphes cubiques sans pont de 10 sommets (dont Petersen)", pop.Count);
    int nSat = 0, nSnark = 0;
    foreach (var g in pop)
    {
        var (col, statut) = Resolveur.ColorationAretesZ3(g, 3, 10.0);
        if (col != null) { nSat++; continue; }
        if (statut == "UNSAT")
        {
            nSnark++;
            bool iso = Analyse.EstIsomorphe(g, PETERSEN);
            Console.WriteLine("  snark trouve -> isomorphe a Petersen : {0}", iso);
            if (!iso) throw new Exception("snark non isomorphe a Petersen !");
        }
    }
    Console.WriteLine("3-colorables : {0} | snarks (INFEASIBLE) : {1} | tous isomorphes a Petersen : OK", nSat, nSnark);
}


population : 16 graphes cubiques sans pont de 10 sommets (dont Petersen)


  snark trouve -> isomorphe a Petersen : True


3-colorables : 15 | snarks (INFEASIBLE) : 1 | tous isomorphes a Petersen : OK


La population confirme le fait classique : **le seul** snark de 10 sommets est le graphe de Petersen. Sur l'ensemble des graphes cubiques sans pont de 10 sommets testés (un lot aléatoire **plus** Petersen lui-même), chaque graphe que le solveur déclare INFEASIBLE à $k=3$ est **isomorphe** à Petersen — le test `EstIsomorphe` rend `True` à chaque fois, et un seul contre-exemple non isomorphe aurait fait échouer la cellule.

C'est la forme empirique du théorème « le seul snark d'ordre 10 est Petersen » : les 10 sommets peuvent être permutés, la structure est toujours la même.

### Exercice 2 (à compléter) : l'extension d'un sommet en triangle

**Énoncé** : l'**extension de Tietze** remplace un sommet $v$ de degré 3 par un **triangle** de trois sommets, reliés chacun à un voisin distinct de $v$. Écrire `EtendreSommetTriangle(g, v)` qui renvoie le graphe étendu.

**Indice** : un sommet du triangle a déjà **deux** voisins internes (les deux autres sommets du triangle) ; il lui faut **exactement un** voisin externe pour que le graphe reste cubique.

**Pourquoi** : l'extension de Tietze **préserve** la 3-colorabilité d'arêtes — c'est l'outil avec lequel on construit de **nouveaux** snarks à partir de Petersen (en étendant un sommet, le graphe reste INFEASIBLE). L'exercice est le premier pas vers la génération de snarks de taille supérieure.

In [16]:
{
    Graphe EtendreSommetTriangle(Graphe g, int v)
    {
        // TODO: etendez le sommet v de G en un triangle
        // Etape 1 : copiez G (construisez un Graphe vide, recopiez aretes et noms)
        // Etape 2 : relevez les trois voisins de v, supprimez v
        // Etape 3 : ajoutez le triangle x-y-z, reliez chaque ancien voisin a un sommet distinct
        // Indice : un sommet du triangle a deja deux voisins internes, il lui faut
        //          exactement un voisin externe pour que le graphe reste cubique
        Console.WriteLine("Exercice a completer");
        return null;
    }

    EtendreSommetTriangle(PETERSEN, 0);

    // Tests attendus (decommentez apres implementation) :
    // var tietze = EtendreSommetTriangle(PETERSEN, 0);
    // var d = Analyse.Decrire("Tietze", tietze);
    // var (col3t, st3) = Resolveur.ColorationAretesZ3(tietze, 3, 10.0);
    // var (col4t, st4) = Resolveur.ColorationAretesZ3(tietze, 4, 10.0);
    // Console.WriteLine("k=3 : {0} | k=4 : {1}", st3, st4);
}


Exercice a completer


<null>

### Exercice 3 (résolu) : explorer la famille de Möbius

**Énoncé** : faire varier la taille $n$ des échelles de Möbius et relever, pour chaque $n$ : la planarité, l'apex, et la 3-colorabilité.

**Méthode** : parcourir une liste de tailles, construire $M_n$, et appeler `Analyse.Decrire` (planarité + apex) et le solveur Z3 (3-colorabilité). Rassembler le tout dans un tableau.

L'exploration sur les grandes tailles ($n$ jusqu'à 40) confirme la règle : **toutes** les échelles de Möbius sont non planaires, apex, et 3-colorables — le théorème apex-2026 tient sur toute la famille, à toute taille.

In [17]:
{
    // Exploration des echelles de Mobius plus grandes (n = 14..40).
    (int, bool, bool, bool, string)[] ExplorerMobius(int[] tailles)
    {
        var resultats = new List<(int, bool, bool, bool, string)>();
        foreach (int n in tailles)
        {
            if (n % 2 != 0) continue;
            var g = Graphes.EchelleMobius(n);
            var d = Analyse.Decrire("M" + n, g);
            var (coloration, statut) = Resolveur.ColorationAretesZ3(g, 3, 10.0);
            resultats.Add((n, d.Planaire, d.Apex, coloration != null, statut));
        }
        return resultats.ToArray();
    }

    var dfGrand = ExplorerMobius(new[] { 14, 16, 18, 20, 30, 40 });
    Console.WriteLine("{0,4}  {1,9}  {2,5}  {3,11}  statut", "n", "planaire", "apex", "3-colorable");
    foreach (var (n, pl, ap, c3, statut) in dfGrand)
    {
        Console.WriteLine("{0,4}  {1,9}  {2,5}  {3,11}  {4}",
            n, Affichage.V(pl), Affichage.V(ap), Affichage.V(c3), statut);
    }
}


   n   planaire   apex  3-colorable  statut


  14      False   True         True  SAT


  16      False   True         True  SAT


  18      False   True         True  SAT


  20      False   True         True  SAT


  30      False   True         True  SAT


  40      False   True         True  SAT


<null>

Le tableau confirme la **règle générale** sur toute la famille : non planaire, apex, 3-colorable. L'apex est **stable** sous le changement d'échelle : supprimer un sommet d'une échelle de Möbius laisse toujours une échelle (ordinaire) planaire.

C'est la famille sur laquelle le théorème 2026 gagne le plus : elle est **toute** dans la zone « apex non planaire », et **toute** 3-colorable. Un explorateur du théorème a ici sa salle de jeu naturelle.

### Exercice 3 (à compléter) : la réciproque de l'apex

**Énoncé** : la réciproque de l'apex est-elle vraie ? C'est-à-dire : un graphe cubique **3-colorable** est-il nécessairement **apex** ? Écrire `ClasserFamille(graphes)` qui, pour une famille de graphes, classe chacun par (planaire, apex, 3-colorable) et fait apparaître les **non-apex qui restent 3-colorables**.

**Indice** : un tel graphe ne met **pas** le théorème en défaut — l'implication ne va que dans un sens (apex $\Rightarrow$ 3-colorable, pas l'inverse). Pensez au graphe de **Heawood** (l'incidence de Fano) : il est 3-colorable sans être apex.

**Attendu** : la famille de test (K33, cube Q3, M8, Petersen, Heawood) fait apparaître au moins un non-apex 3-colorable — le Heawood — qui illustre que l'apex est une **condition suffisante**, pas nécessaire.

In [18]:
{
    (string, bool, bool, bool, string)[] ClasserFamille((string, Graphe)[] graphes)
    {
        // TODO: classez chaque graphe de la famille (planaire / apex / 3-colorable)
        // Etape 1 : appelez Analyse.Decrire(nom, g) pour les proprietes structurelles
        // Etape 2 : appelez Resolveur.ColorationAretesZ3(g, 3, 10.0) et relevez le statut
        // Etape 3 : rassemblez, puis identifiez les non-apex qui restent 3-colorables
        // Indice : un tel graphe ne met PAS le theoreme en defaut -- l'implication ne
        //          va que dans un sens ; pensez au graphe de Heawood
        Console.WriteLine("Exercice a completer");
        return null;
    }

    ClasserFamille(Array.Empty<(string, Graphe)>());

    // Constructeurs de la famille attendue (decommentez apres implementation) :
    // var famille = new (string, Graphe)[]
    // {
    //     ("K33", Graphes.CompletBiparti(3, 3)),
    //     ("cube Q3", Graphes.Hypercube3()),
    //     ("M8", Graphes.EchelleMobius(8)),
    //     ("Petersen", PETERSEN),
    //     ("Heawood", Graphes.Heawood()),
    // };
    // var classement = ClasserFamille(famille);
    // foreach (var (nom, pl, ap, c3, statut) in classement)
    //     Console.WriteLine("{0,-10} planaire={1} apex={2} 3-colorable={3} statut={4}",
    //         nom, Affichage.V(pl), Affichage.V(ap), Affichage.V(c3), statut);
}


Exercice a completer


<null>

***

## 7. Conclusion (~3 min)

Ce notebook a vérifié **empiriquement**, avec le solveur Z3 en C#, les deux théorèmes récents sur la 3-coloration d'arêtes des graphes cubiques :

- **sans-pont-2026** : le **pont** est l'obstacle — le cubique à pont (planaire, apex) est INFEASIBLE à $k=3$, et l'argument de parité l'explique ;
- **apex-2026** : la zone « apex mais non planaire » (les échelles de Möbius) est bien 3-colorable, au-delà du périmètre de Tutte-1966.

Le **graphe de Petersen** reste le contre-exemple central : snark minimal, INFEASIBLE à $k=3$, 4-colorable — le point où la conjecture de Tutte **s'arrête**.

### Ce que le portage C# apporte

- **Un solveur SOTA différent** : Z3 (SMT généraliste) remplace CP-SAT (optimisation combinatoire dédiée) — la même instance CSP, formulée en théorie booléenne pure ;
- **Des implémentations du carnet** : les outils networkx (ponts, planarité, échantillonnage aléatoire) sont portés en C# — le carnet est **autonome**, sans dépendance réseau au-delà de NuGet ;
- **La même pédagogie** : les trois exercices (indice chromatique, classes de Vizing / snarks, apex / Möbius) sont inchangés, seuls les langages des exemples résolus changent.

Le lecteur a maintenant **deux** implementations du même résultat — Python (CP-SAT + networkx) et C# (Z3 + implémentations du carnet) — et peut choisir l'outillage qui l'arrange. Le **Rust** (troisième version) complète le trio avec rustsat/batsat.

**Pistes de suite** :

1. **Prouver** (pas seulement vérifier) : brancher un assistant de preuve (Lean 4) sur la 3-coloration pour transformer la vérification empirique en preuve formelle ;
2. **Générer des snarks** : l'extension de Tietze (exercice 2) comme outil de construction systématique de snarks au-delà de Petersen ;
3. **Comparer les solveurs** : mesurer frontalement CP-SAT vs Z3 vs rustsat sur une suite de snarks croissants — où chacun des trois outillages prend le dessus.
